In [ ]:
import os
import sqlite3
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DB_PATH = os.path.join(BASE_DIR, "codecity.db")
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

conn = sqlite3.connect(DB_PATH)
files_df = pd.read_sql_query(
    "SELECT snapshot_id, path, name, size, complexity, width, depth, height, is_test_file, area, aspect_ratio FROM files",
    conn,
)
conn.close()

# Simple synthetic label: mark top 20% complexity files as high risk
threshold = files_df["complexity"].quantile(0.8)
files_df["is_high_risk"] = (files_df["complexity"] >= threshold).astype(int)

feature_cols = [
    "size",
    "complexity",
    "width",
    "depth",
    "height",
    "is_test_file",
    "area",
    "aspect_ratio",
]

X = files_df[feature_cols].values
y = files_df["is_high_risk"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

model_path = os.path.join(MODELS_DIR, "risk_model.joblib")
meta_path = os.path.join(MODELS_DIR, "risk_model_meta.json")

joblib.dump(clf, model_path)

with open(meta_path, "w") as f:
    json.dump({"feature_cols": feature_cols}, f, indent=2)

print("Saved model to", model_path)
print("Saved metadata to", meta_path)